# Setup

In [ ]:
import os
import sys

import pandas as pd
from processor.main import Processor
from processor.table.representation.impl.df_table import DFTable
from processor.table.store.table_store_factory import ImplementedTableStore

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("."), "src")))

In [ ]:
# OpenAI model
# from dotenv import load_dotenv
# load_dotenv()
# api_key = os.getenv('OPENAI_API_KEY')
# model = 'gpt-4o-mini-2024-07-18'

# Local Model
model = "src/processor/llm/weight/qwen25-7b"

embed_path = "src/processor/llm/weight/bge-base"
processor = Processor(
    model, embed_path, ImplementedTableStore.PY_TABLE_STORE, "processor_output"
)

In [ ]:
# Define CONSTANTS
DB_SCHEMA = "E2E_SCHEMA"
QUESTION = "Which restaurant has ratings above 3.0?"

In [ ]:
# Index sample tables
zomato = pd.read_csv("data_src/zomato.csv")
yelp = pd.read_csv("data_src/yelp.csv")

processor.ctx.table_store.create_db_schema(DB_SCHEMA)
processor.ctx.table_store.add_table(
    DB_SCHEMA,
    "zomato",
    DFTable(zomato),
)
processor.ctx.table_store.add_table(
    DB_SCHEMA,
    "yelp",
    DFTable(yelp),
)

# Step 1: Schema Enhancement & Target Schema Generation

In [6]:
target_schema_node = processor.get_target_schema(QUESTION)
target_schema = target_schema_node.computation_output
print(target_schema)

[2025-04-20 10:26:53] INFO in schema_processor: Getting target schema for the question Which restaurant has ratings above 3.0?


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[2025-04-20 10:27:05] INFO in schema_processor: => Target schema: ['Restaurant ID', 'Restaurant Name', 'Rating']
['Restaurant ID', 'Restaurant Name', 'Rating']


In [ ]:
table_descriptions_node = processor.get_table_descriptions(DB_SCHEMA)
table_descriptions = table_descriptions_node.computation_output
print(table_descriptions)

Describing tables:   0%|          | 0/2 [00:00<?, ?it/s]

[2025-04-20 10:27:05] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-04-20 10:27:25] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-04-20 10:27:32] INFO in schema_processor: => Overall description of table 'zomato': The table likely represents information about restaurants, including their ID, name, average rating, phone number, number of reviews, and address.


Describing tables:  50%|█████     | 1/2 [00:26<00:26, 26.87s/it]

[2025-04-20 10:27:32] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-04-20 10:27:49] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-04-20 10:27:53] INFO in schema_processor: => Overall description of table 'yelp': The table likely represents restaurant information, including ratings, contact details, and addresses.


Describing tables: 100%|██████████| 2/2 [00:47<00:00, 23.95s/it]

{'zomato': 'The table likely represents information about restaurants, including their ID, name, average rating, phone number, number of reviews, and address.', 'yelp': 'The table likely represents restaurant information, including ratings, contact details, and addresses.'}


In [8]:
enhanced_schemas_node = processor.get_enhanced_schemas(
    DB_SCHEMA, table_descriptions, 3, [table_descriptions_node]
)
enhanced_schemas = enhanced_schemas_node.computation_output
print(enhanced_schemas)

[2025-04-20 10:27:54] INFO in schema_processor: Enhancing the schema of table 'zomato'
[2025-04-20 10:27:54] INFO in schema_processor: => Table description: The table likely represents information about restaurants, including their ID, name, average rating, phone number, number of reviews, and address.
[2025-04-20 10:27:54] INFO in schema_processor: => Schema before enhancement: col: ID | NAME | RATING | PHONENUMBER | NO_OF_REVIEWS | ADDRESS
[2025-04-20 10:27:54] INFO in schema_processor: ==> Renaming column `ID`
[2025-04-20 10:28:08] INFO in schema_processor: ==> New column name RESTAURANT_ID
[2025-04-20 10:28:08] INFO in schema_processor: ==> Renaming column `NAME`
[2025-04-20 10:28:22] INFO in schema_processor: ==> New column name RestaurantName
[2025-04-20 10:28:22] INFO in schema_processor: ==> Renaming column `RATING`
[2025-04-20 10:28:42] INFO in schema_processor: ==> New column name Average_Rating
[2025-04-20 10:28:42] INFO in schema_processor: ==> Renaming column `PHONENUMBER`

# Visualization

In [ ]:
from pyvis.network import Network
import networkx as nx

from processor.computation_graph import ComputationGraph


def interactive_network_pyvis(graph: ComputationGraph):
    G = nx.DiGraph()

    for node in graph.nodes:
        # label = f"{node.function_name}()\n{node.class_name or ''}\n{node.computation_description}"
        label = f"{node.function_name}()"
        G.add_node(node.id, label=label)

    for node in graph.nodes:
        for input_node in node.input_nodes:
            G.add_edge(input_node.id, node.id)

    net = Network(
        notebook=True,
        height="600px",
        width="100%",
        directed=True,
        cdn_resources="in_line",
    )
    net.from_nx(G)
    net.show("graph.html")  # Will now render inline in Jupyter

In [38]:
interactive_network_pyvis(processor.ctx.computation_graph)

graph.html
